# Libraries


In [1]:
import os
import math
import requests
import pandas as pd
import json, ast
from datetime import datetime, timedelta
import time
import paho.mqtt.client as mqtt

# 1. DATA RETRIEVAL

## 1.1. Declaring Constants 

In [47]:
# ADD API KEY HERE
API_KEY="oe_3ZSXpKzAEksXKrKt9BKzZDzW" # API Key please use mindfully

In [48]:
#Configs
BASE_URL = "https://api.openelectricity.org.au/v4"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
INTERVAL = "5m"

# Date range for 1st week of October 2025, End date is not inclusive.
START_DATE = datetime(2025, 10, 1)
END_DATE = datetime(2025, 10, 8)
REQUEST_DELAY = 60  # seconds between requests to avoid API limits

## 1.2. Fetching the list of Facilities

Fetch the list of all facilities for a given electricity network (default=NEM),  
Returns: pd.DataFrame: Facility metadata including name, region, code, location, and units.

In [49]:
def fetch_all_facilities(network: str = "NEM") -> pd.DataFrame:
    
    url = f"{BASE_URL}/facilities/"
    params = {
        "network_code": "NEM",
    }

    print("IN PROGRESS Fetching all facilities...")
    r = requests.get(url, headers=HEADERS, params=params)
    if r.status_code >= 400:
        print("API Error:")
        try:
            print(r.json())
        except Exception:
            print(r.text)
        r.raise_for_status()

    data = r.json().get("data", [])
    rows = []
    for entry in data:
        rows.append({
            "name": entry.get("name"),
      "network_id": entry.get("network_id"),
      "network_region": entry.get("network_region"),
      "description": entry.get("description"),
      "location": entry.get("location"),
       "code": entry.get("code"),

            "units": entry.get("units"),
        
          "created_at": entry.get("created_at"),
          "updated_at": entry.get("updated_at"),
       
       
        })

    df = pd.DataFrame(rows)
    print(f"SUCCESS Retrieved {len(df)} facilities")
    return df

facilities_df = fetch_all_facilities()


IN PROGRESS Fetching all facilities...
SUCCESS Retrieved 580 facilities


### 1.2.1. PER-FACILITY POWER GENERATED (5 min interval)

fetches 5-minute interval power generation data for every facility in the NEM network.  

In [50]:
#fetches 5-minute interval power generation data for every facility in the NEM network.  
def fetch_facility_power(facility_id: str, start: datetime, end: datetime) -> pd.DataFrame:
    """Fetch 5-minute power data for a single facility and flatten it."""
    url = f"{BASE_URL}/data/facilities/NEM"
    #setting request parameters 
    params = {
        "metrics": ["power"],
        "interval": INTERVAL,
        "facility_code": facility_id,
        "date_start": START_DATE,
        "date_end": END_DATE,
        
    }

    # Make request and handle API / network issues safely
    try:
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        json_data = r.json()
    except Exception as e:
        print(f" Failed to fetch power for {facility_id}: {e}")
        return pd.DataFrame()

    # Check if the API returned success
    if not json_data.get("success", False):
        print(f" API returned failure for {facility_id}: {json_data.get('error')}")
        return pd.DataFrame()

    data = json_data.get("data", [])
    rows = []

    # Flatten nested JSON:
    for entry in data:
        unit_code = entry.get("unit", "MW")
        for res in entry.get("results", []):
            for point in res.get("data", []):
                timestamp, value = point
                rows.append({
                    "facility_id": facility_id,
                    "unit_code": unit_code,
                    "timestamp": timestamp,
                    "power_MW": value
                })

    return pd.DataFrame(rows)



# Step 1: Load facilities CSV (if needed)
facility_ids = facilities_df["code"]
print(f"IN PROGRESS Fetching power for {len(facility_ids)} facilities...")

all_power = []
count =1
# Step 2: Iterate over facilities
for facility_id in facility_ids:
    print(f"Fetching power for facility {facility_id}...count no "+ str(count))
    count+=1
    power_df = fetch_facility_power(facility_id, START_DATE, END_DATE)
    if power_df.empty:
        print(f" No data for {facility_id}")
    all_power.append(power_df)
    time.sleep(REQUEST_DELAY)

# Step 3: Combine all facilities
power_df_raw = pd.concat(all_power, ignore_index=True)


IN PROGRESS Fetching power for 580 facilities...
Fetching power for facility ADP...count no 1
Fetching power for facility ALBANY...count no 2
⚠️ Failed to fetch power for ALBANY: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=ALBANY&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for ALBANY
Fetching power for facility ALDGASF...count no 3
Fetching power for facility AMCORGR...count no 4
⚠️ Failed to fetch power for AMCORGR: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=AMCORGR&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for AMCORGR
Fetching power for facility ANGASTON...count no 5
Fetching power for facility APS...count no 6
⚠️ Failed to fetch power for APS: 416 Client Error: Range Not Satisfiable for url: https://api.opene

⚠️ Failed to fetch power for 0BSSF: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=0BSSF&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for 0BSSF
Fetching power for facility BWTR1...count no 55
Fetching power for facility BHILLGT...count no 56
Fetching power for facility BROKENH...count no 57
Fetching power for facility BHB...count no 58
Fetching power for facility BROOKLYN...count no 59
⚠️ Failed to fetch power for BROOKLYN: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=BROOKLYN&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for BROOKLYN
Fetching power for facility BROWNMT...count no 60
⚠️ Failed to fetch power for BROWNMT: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities

⚠️ Failed to fetch power for COLLIE_ESR: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=COLLIE_ESR&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for COLLIE_ESR
Fetching power for facility COLLIE_BESS2...count no 106
⚠️ Failed to fetch power for COLLIE_BESS2: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=COLLIE_BESS2&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for COLLIE_BESS2
Fetching power for facility COLNSV...count no 107
⚠️ Failed to fetch power for COLNSV: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=COLNSV&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for COLNSV
Fetching power for fa

⚠️ Failed to fetch power for 0FULHAMSF: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=0FULHAMSF&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for 0FULHAMSF
Fetching power for facility GANGARR...count no 160
Fetching power for facility GANNBESS...count no 161
Fetching power for facility GANNSF...count no 162
Fetching power for facility GEORGTWN...count no 163
⚠️ Failed to fetch power for GEORGTWN: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=GEORGTWN&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for GEORGTWN
Fetching power for facility TESLA_GERALDTON...count no 164
⚠️ Failed to fetch power for TESLA_GERALDTON: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metric

Fetching power for facility HASTING...count no 204
⚠️ Failed to fetch power for HASTING: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=HASTING&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for HASTING
Fetching power for facility HAUGHT1...count no 205
Fetching power for facility HD1WF...count no 206
Fetching power for facility HAYMSF...count no 207
Fetching power for facility HAZEL...count no 208
⚠️ Failed to fetch power for HAZEL: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=HAZEL&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for HAZEL
Fetching power for facility HBESS...count no 209
Fetching power for facility HENDERSON_RENEWABLE...count no 210
⚠️ Failed to fetch power for HENDERSON_RENEWABLE: 416 Client Error: Range No

...

⚠️ Failed to fetch power for MM: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=MM&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for MM
Fetching power for facility MUCRKSF...count no 347
Fetching power for facility MUWAWF...count no 348
Fetching power for facility MURRAY...count no 349
Fetching power for facility MBPS2...count no 350
Fetching power for facility MUSSELRO...count no 351
Fetching power for facility NASF...count no 352
⚠️ Failed to fetch power for NASF: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=NASF&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for NASF
Fetching power for facility NEWGEN_NEERABUP...count no 353
⚠️ Failed to fetch power for NEWGEN_NEERABUP: 416 Client Error: Range Not Satisfiable for url: ht

Fetching power for facility STANVAC...count no 391
Fetching power for facility PORTWF...count no 392
Fetching power for facility QUARANTN...count no 393
Fetching power for facility QUERIVER...count no 394
⚠️ Failed to fetch power for QUERIVER: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=QUERIVER&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for QUERIVER
Fetching power for facility QBYNB...count no 395
Fetching power for facility QP...count no 396
⚠️ Failed to fetch power for QP: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=QP&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for QP
Fetching power for facility RACOMIL...count no 397
⚠️ Failed to fetch power for RACOMIL: 416 Client Error: Range Not Satisfiable for url: https:

⚠️ Failed to fetch power for SLDCBLK: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=SLDCBLK&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for SLDCBLK
Fetching power for facility 0STANBESS...count no 444
⚠️ Failed to fetch power for 0STANBESS: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=0STANBESS&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for 0STANBESS
Fetching power for facility STANWELL...count no 445
Fetching power for facility STAPYLTON...count no 446
⚠️ Failed to fetch power for STAPYLTON: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=STAPYLTON&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ 

⚠️ Failed to fetch power for 0TILBSF: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=0TILBSF&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for 0TILBSF
Fetching power for facility TIMWEST...count no 487
⚠️ Failed to fetch power for TIMWEST: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=TIMWEST&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for TIMWEST
Fetching power for facility 0TOMAGOBESS...count no 488
⚠️ Failed to fetch power for 0TOMAGOBESS: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=0TOMAGOBESS&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for 0TOMAGOBESS
Fetching power for facility TO

Fetching power for facility WDBESS...count no 537
Fetching power for facility WDGPH...count no 538
Fetching power for facility WDBESS2...count no 539
Fetching power for facility WESTCBT...count no 540
⚠️ Failed to fetch power for WESTCBT: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=WESTCBT&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for WESTCBT
Fetching power for facility WRSF1...count no 541
Fetching power for facility WRWF1...count no 542
Fetching power for facility WHITSF...count no 543
Fetching power for facility WHIT1...count no 544
⚠️ Failed to fetch power for WHIT1: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=power&interval=5m&facility_code=WHIT1&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for WHIT1
Fetching power for facility WILGAPK...co

### 1.2.2. PER-FACILITY CO2 EMISSIONS (5 min interval)


fetches 5-minute interval emnission data for every facility in the NEM network.  

In [51]:
#Fetches 5-minute emissions data for a single facility and returns a flattened DataFrame.
def fetch_facility_emissions(facility_id: str, start: datetime, end: datetime) -> pd.DataFrame:
    """Fetch 5-minute emissions data for a single facility and flatten it."""
    url = f"{BASE_URL}/data/facilities/NEM"
    # Passing Parameters
    params = {
        "metrics": ["emissions"],
        "interval": INTERVAL,
        "facility_code": facility_id,
        "date_start": START_DATE,
        "date_end": END_DATE,
    }

    # Make request and safely handle connection / API issues
    try:
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        json_data = r.json()
    except Exception as e:
        print(f" Failed to fetch emissions for {facility_id}: {e}")
        return pd.DataFrame()

    # Check API success flag
    if not json_data.get("success", False):
        print(f" API returned failure for {facility_id}: {json_data.get('error')}")
        return pd.DataFrame()

    data = json_data.get("data", [])
    rows = []

    # Flatten nested structure
    for entry in data:
        unit_code = entry.get("unit", "tCO2e")  # adjusted unit label
        for res in entry.get("results", []):
            for point in res.get("data", []):
                timestamp, value = point
                rows.append({
                    "facility_id": facility_id,
                    "unit_code": unit_code,
                    "timestamp": timestamp,
                    "emission": value
                })

    return pd.DataFrame(rows)


print(f"IN PROGRESS Fetching emissions for {len(facility_ids)} facilities...")

all_emissions = []

# Step 2: Iterate over facilities
count = 1
for facility_id in facility_ids:
    print(f"Fetching emissions for facility {facility_id}... count no {count}")
    count += 1
    emissions_df = fetch_facility_emissions(facility_id, START_DATE, END_DATE)
    if emissions_df.empty:
        print(f" No data for {facility_id}")
    all_emissions.append(emissions_df)
    time.sleep(REQUEST_DELAY)

# Step 3: Combine all facilities
emissions_df_raw = pd.concat(all_emissions, ignore_index=True)



IN PROGRESS Fetching emissions for 580 facilities...
Fetching emissions for facility ADP... count no 1
Fetching emissions for facility ALBANY... count no 2
⚠️ Failed to fetch emissions for ALBANY: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=ALBANY&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for ALBANY
Fetching emissions for facility ALDGASF... count no 3
Fetching emissions for facility AMCORGR... count no 4
⚠️ Failed to fetch emissions for AMCORGR: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=AMCORGR&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for AMCORGR
Fetching emissions for facility ANGASTON... count no 5
Fetching emissions for facility APS... count no 6
⚠️ Failed to fetch emissions for APS: 416 Client E

Fetching emissions for facility BROADMDW... count no 53
⚠️ Failed to fetch emissions for BROADMDW: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=BROADMDW&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for BROADMDW
Fetching emissions for facility 0BSSF... count no 54
⚠️ Failed to fetch emissions for 0BSSF: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=0BSSF&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for 0BSSF
Fetching emissions for facility BWTR1... count no 55
Fetching emissions for facility BHILLGT... count no 56
Fetching emissions for facility BROKENH... count no 57
Fetching emissions for facility BHB... count no 58
Fetching emissions for facility BROOKLYN... count no 59
⚠️ Failed to fetch emissions for BROOKLY

Fetching emissions for facility COLWF01... count no 101
Fetching emissions for facility INVESTEC_COLLGAR... count no 102
⚠️ Failed to fetch emissions for INVESTEC_COLLGAR: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=INVESTEC_COLLGAR&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for INVESTEC_COLLGAR
Fetching emissions for facility COLLIE... count no 103
⚠️ Failed to fetch emissions for COLLIE: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=COLLIE&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for COLLIE
Fetching emissions for facility COLLIE_ESR4... count no 104
⚠️ Failed to fetch emissions for COLLIE_ESR4: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?met

Fetching emissions for facility ERB... count no 151
Fetching emissions for facility ERGT01... count no 152
Fetching emissions for facility 0ERB2... count no 153
⚠️ Failed to fetch emissions for 0ERB2: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=0ERB2&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for 0ERB2
Fetching emissions for facility FWF... count no 154
⚠️ Failed to fetch emissions for FWF: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=FWF&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for FWF
Fetching emissions for facility FINLEYSF... count no 155
Fetching emissions for facility FISHER... count no 156
Fetching emissions for facility FLATROCKS... count no 157
⚠️ Failed to fetch emissions for FLATROCKS: 416 Cli

Fetching emissions for facility GNNDHSF... count no 192
Fetching emissions for facility GUNNING... count no 193
Fetching emissions for facility 0GUNSYNDSF... count no 194
⚠️ Failed to fetch emissions for 0GUNSYNDSF: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=0GUNSYNDSF&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for 0GUNSYNDSF
Fetching emissions for facility SNOWY3... count no 195
Fetching emissions for facility HLMSEW01... count no 196
⚠️ Failed to fetch emissions for HLMSEW01: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=HLMSEW01&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for HLMSEW01
Fetching emissions for facility HALAMRD... count no 197
⚠️ Failed to fetch emissions for HALAMRD: 416 Client Error: Range

...

⚠️ Failed to fetch emissions for PEDLER: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=PEDLER&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for PEDLER
Fetching emissions for facility NPPPPS... count no 377
Fetching emissions for facility PIBESS... count no 378
Fetching emissions for facility PHOENIX_KWINANA_WTE... count no 379
⚠️ Failed to fetch emissions for PHOENIX_KWINANA_WTE: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=PHOENIX_KWINANA_WTE&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for PHOENIX_KWINANA_WTE
Fetching emissions for facility TESLA_PICTON... count no 380
⚠️ Failed to fetch emissions for TESLA_PICTON: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facil

Fetching emissions for facility RYANCWF... count no 418
Fetching emissions for facility RYEPARK... count no 419
Fetching emissions for facility SALTCRK... count no 420
Fetching emissions for facility SAPHWF1... count no 421
Fetching emissions for facility SEBSF... count no 422
Fetching emissions for facility SHEP1... count no 423
⚠️ Failed to fetch emissions for SHEP1: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=SHEP1&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for SHEP1
Fetching emissions for facility SHOALHAV... count no 424
Fetching emissions for facility SHOAL... count no 425
Fetching emissions for facility STWF... count no 426
Fetching emissions for facility SITHE... count no 427
Fetching emissions for facility SMTHBES1... count no 428
Fetching emissions for facility SNAPPER... count no 429
Fetching emissions for facility SNOWTOWN... count n

Fetching emissions for facility TB2SF... count no 465
Fetching emissions for facility TALLAWAR... count no 466
Fetching emissions for facility TALWB... count no 467
Fetching emissions for facility TAMALA_PARK... count no 468
⚠️ Failed to fetch emissions for TAMALA_PARK: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=TAMALA_PARK&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for TAMALA_PARK
Fetching emissions for facility TVCCPS... count no 469
Fetching emissions for facility TARALGA... count no 470
Fetching emissions for facility 0TARONGBESS... count no 471
Fetching emissions for facility TARONG... count no 472
Fetching emissions for facility TARONGN... count no 473
Fetching emissions for facility TARRALEA... count no 474
Fetching emissions for facility TATIARA... count no 475
⚠️ Failed to fetch emissions for TATIARA: 416 Client Error: Range Not Satisf

⚠️ Failed to fetch emissions for ALCOA_WGP: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=ALCOA_WGP&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for ALCOA_WGP
Fetching emissions for facility WAGGNSF... count no 511
Fetching emissions for facility ALINTA_WWF... count no 512
⚠️ Failed to fetch emissions for ALINTA_WWF: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=ALINTA_WWF&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for ALINTA_WWF
Fetching emissions for facility WLWLSF... count no 513
Fetching emissions for facility WW... count no 514
⚠️ Failed to fetch emissions for WW: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facili

Fetching emissions for facility WIVENHOE... count no 555
Fetching emissions for facility WIVENSH... count no 556
⚠️ Failed to fetch emissions for WIVENSH: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=WIVENSH&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for WIVENSH
Fetching emissions for facility WOLARSF... count no 557
Fetching emissions for facility WOLLERT... count no 558
⚠️ Failed to fetch emissions for WOLLERT: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facilities/NEM?metrics=emissions&interval=5m&facility_code=WOLLERT&date_start=2025-10-01+00%3A00%3A00&date_end=2025-10-08+00%3A00%3A00
⚠️ No data for WOLLERT
Fetching emissions for facility WONWP... count no 559
⚠️ Failed to fetch emissions for WONWP: 416 Client Error: Range Not Satisfiable for url: https://api.openelectricity.org.au/v4/data/facil

### 1.2.3. PER-MARKET POWER PRICE AND DEMAND (5 min interval)


fetches 5-minute market data (spot price and demand) for each network region of the NEM.

In [52]:
NETWORK = "NEM"
NETWORK_REGIONS = ["NSW1", "QLD1", "SA1", "TAS1", "VIC1"]

# SETTINGS
NETWORK_REGIONS = ["NSW1", "QLD1", "SA1", "TAS1", "VIC1"]
METRICS = ["price", "demand"]

def fetch_market_metric(network: str, region: str, metric: str, start: datetime, end: datetime) -> pd.DataFrame:
    
    url = f"{BASE_URL}/market/network/{network}"
    params = {
        "metrics": metric,
        "interval": INTERVAL,
        "network_region": region,
        "date_start": START_DATE,
        "date_end": END_DATE,
    }

    # Request + error handling
    try:
        r = requests.get(url, headers=HEADERS, params=params)
        r.raise_for_status()
        json_data = r.json()
    except Exception as e:
        print(f" Failed {metric.upper()} {region} {start.date()}: {e}")
        return pd.DataFrame()

    # API error response handling
    if not json_data.get("success", False):
        print(f" API returned failure for {metric.upper()} {region}: {json_data.get('error')}")
        return pd.DataFrame()

    # Flatten nested JSON
    rows = []
    for entry in json_data.get("data", []):
        unit = entry.get("unit", "")
        for res in entry.get("results", []):
            for point in res.get("data", []):
                if len(point) == 2:
                    timestamp, value = point
                    rows.append({
                        "timestamp": timestamp,
                        "network": network,
                        "network_region": region,
                        f"value_{metric}": value,
                        f"unit_{metric}": unit,
                    })

    df = pd.DataFrame(rows)
    print(f"SUCCESSFUL {region}: {metric.upper()} {start.date()} → {len(df)} rows")
    return df



# Collect one day at a time to avoid overwhelming API
all_data = []
current_date = START_DATE

while current_date < END_DATE:
    next_date = current_date + timedelta(days=1)

    for region in NETWORK_REGIONS:
        region_data = []
        for metric in METRICS:
            df = fetch_market_metric(NETWORK, region, metric, current_date, next_date)
            if not df.empty:
                region_data.append(df)

        # Merge price + demand for this region/day
        if region_data:
            merged_df = region_data[0]
            for df in region_data[1:]:
                merged_df = pd.merge(
                    merged_df,
                    df,
                    on=["timestamp", "network", "network_region"],
                    how="outer"
                )
            all_data.append(merged_df)

    current_date = next_date
    
# Final concatenation of all data
if not all_data:
    print(" No market data retrieved for any region or day.")
 
market_raw = pd.concat(all_data, ignore_index=True)


SUCCESSFUL NSW1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL NSW1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL QLD1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL QLD1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL SA1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL SA1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL TAS1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL TAS1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL VIC1: PRICE 2025-10-01 → 2016 rows
SUCCESSFUL VIC1: DEMAND 2025-10-01 → 2016 rows
SUCCESSFUL NSW1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL NSW1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL QLD1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL QLD1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL SA1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL SA1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL TAS1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL TAS1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL VIC1: PRICE 2025-10-02 → 2016 rows
SUCCESSFUL VIC1: DEMAND 2025-10-02 → 2016 rows
SUCCESSFUL NSW1: PRICE 2025-10-03 → 2016 rows
SUCCESSFUL NSW1: DEMAND 2025

## 1.3. RAW dataframes list

In [53]:
facilities_df

,name,network_id,network_region,description,location,code,units,created_at,updated_at
0,Adelaide Desalination,NEM,SA1,"<p>The Adelaide Desalination plant (ADP), form...","{'lat': -35.096948, 'lng': 138.484061}",ADP,"[{'code': 'ADPPV1', 'fueltech_id': 'solar_util...",2023-10-18T04:34:30Z,2025-08-05T06:08:12Z
1,Albany,WEM,WEM,<p>Albany wind and Grasmere farms are two wind...,"{'lat': -35.065835, 'lng': 117.797506}",ALBANY,"[{'code': 'ALBANY_WF1', 'fueltech_id': 'wind',...",2023-10-18T04:34:30Z,2025-08-11T02:03:13Z
2,Aldoga,NEM,QLD1,<p>The Aldoga Solar Farm will be approximately...,"{'lat': -23.839544, 'lng': 151.0849}",ALDGASF,"[{'code': 'ALDGASF1', 'fueltech_id': 'solar_ut...",2025-01-31T04:19:33Z,2025-03-25T00:52:44Z
3,Amcor Glass,NEM,SA1,<p></p>,"{'lat': -34.882663, 'lng': 138.577975}",AMCORGR,"[{'code': 'AMCORGR', 'fueltech_id': 'distillat...",2023-10-18T04:34:32Z,2023-10-18T04:34:32Z
4,Angaston,NEM,SA1,<p>Angaston Power Station is a diesel-powered ...,"{'lat': -34.503948, 'lng': 139.024296}",ANGASTON,"[{'code': 'ANGAS1', 'fueltech_id': 'distillate...",2023-10-18T04:34:32Z,2025-09-07T01:53:13Z
...,...,...,...,...,...,...,...,...,...
575,Yarrawonga,NEM,VIC1,<p>Yarrawonga Power Station is adjacent to the...,"{'lat': -36.009466, 'lng': 145.999568}",YWNGAHYD,"[{'code': 'YWNGAHYD', 'fueltech_id': 'hydro', ...",2023-10-18T04:35:33Z,2024-10-29T22:19:53Z
576,Yarwun,NEM,QLD1,<p></p>,"{'lat': -23.8302, 'lng': 151.149692}",YARWUN,"[{'code': 'YARWUN_1', 'fueltech_id': 'gas_ccgt...",2023-10-18T04:35:33Z,2025-07-08T04:17:36Z
577,Yatpool,NEM,VIC1,<p>The 258-hectare Yatpool Solar Farm in north...,"{'lat': -34.38073, 'lng': 142.20534}",YATSF1,"[{'code': 'YATSF1', 'fueltech_id': 'solar_util...",2023-10-18T04:35:33Z,2025-07-08T05:20:08Z
578,Yawong,NEM,VIC1,<p>The Yawong Wind Farm was developed and cons...,"{'lat': -36.471022, 'lng': 143.361722}",YAWWF,"[{'code': 'YAWWF1', 'fueltech_id': 'wind', 'st...",2023-10-18T04:35:33Z,2024-10-30T23:59:54Z


In [54]:
power_df_raw 

,facility_id,unit_code,timestamp,power_MW
0,ADP,MW,2025-10-01T00:00:00+10:00,-0.004
1,ADP,MW,2025-10-01T00:05:00+10:00,-0.046
2,ADP,MW,2025-10-01T00:10:00+10:00,0.000
3,ADP,MW,2025-10-01T00:15:00+10:00,0.003
4,ADP,MW,2025-10-01T00:20:00+10:00,-0.018
...,...,...,...,...
979360,YENDONWF,MW,2025-10-07T23:35:00+10:00,88.630
979361,YENDONWF,MW,2025-10-07T23:40:00+10:00,91.890
979362,YENDONWF,MW,2025-10-07T23:45:00+10:00,89.030
979363,YENDONWF,MW,2025-10-07T23:50:00+10:00,85.830


In [55]:
emissions_df_raw 

,facility_id,unit_code,timestamp,emission
0,ADP,t,2025-10-01T00:00:00+10:00,0.0
1,ADP,t,2025-10-01T00:05:00+10:00,0.0
2,ADP,t,2025-10-01T00:10:00+10:00,0.0
3,ADP,t,2025-10-01T00:15:00+10:00,0.0
4,ADP,t,2025-10-01T00:20:00+10:00,0.0
...,...,...,...,...
977344,YENDONWF,t,2025-10-07T23:35:00+10:00,0.0
977345,YENDONWF,t,2025-10-07T23:40:00+10:00,0.0
977346,YENDONWF,t,2025-10-07T23:45:00+10:00,0.0
977347,YENDONWF,t,2025-10-07T23:50:00+10:00,0.0


In [56]:
market_raw

,timestamp,network,network_region,value_price,unit_price,value_demand,unit_demand
0,2025-10-01T00:00:00+10:00,NEM,NSW1,56.98,$/MWh,7105.57,MW
1,2025-10-01T00:05:00+10:00,NEM,NSW1,80.01,$/MWh,7170.68,MW
2,2025-10-01T00:10:00+10:00,NEM,NSW1,58.40,$/MWh,7158.50,MW
3,2025-10-01T00:15:00+10:00,NEM,NSW1,80.01,$/MWh,7177.80,MW
4,2025-10-01T00:20:00+10:00,NEM,NSW1,65.00,$/MWh,7085.54,MW
...,...,...,...,...,...,...,...
70555,2025-10-07T23:35:00+10:00,NEM,VIC1,104.22,$/MWh,4709.39,MW
70556,2025-10-07T23:40:00+10:00,NEM,VIC1,103.68,$/MWh,4696.86,MW
70557,2025-10-07T23:45:00+10:00,NEM,VIC1,95.86,$/MWh,4639.93,MW
70558,2025-10-07T23:50:00+10:00,NEM,VIC1,95.88,$/MWh,4616.01,MW


# 2. DATA INTEGRATION/CLEANING

## 2.1. Cleaning and Combining Power-Emission

Merge facility power and emissions data on facility ID and timestamp. Outer join keeps rows even if power or emissions is missing for a time period

In [57]:
power_emission_raw_stage_1 = pd.merge(
    power_df_raw,
    emissions_df_raw,
    on=['facility_id', 'timestamp'],
    how='outer' 
).sort_values(by=['timestamp','facility_id']).reset_index(drop=True)
print(power_emission_raw_stage_1)

        facility_id unit_code_x                  timestamp  power_MW  \
0             0MREH          MW  2025-10-01T00:00:00+10:00      0.00   
1             0MREH          MW  2025-10-01T00:00:00+10:00      0.00   
2             0MREH          MW  2025-10-01T00:00:00+10:00      0.00   
3             0MREH          MW  2025-10-01T00:00:00+10:00      0.00   
4           0MREHA2          MW  2025-10-01T00:00:00+10:00      0.00   
...             ...         ...                        ...       ...   
1930540     YARANSF          MW  2025-10-07T23:55:00+10:00      0.00   
1930541      YARWUN          MW  2025-10-07T23:55:00+10:00    138.99   
1930542      YATSF1          MW  2025-10-07T23:55:00+10:00      0.00   
1930543    YENDONWF          MW  2025-10-07T23:55:00+10:00     89.87   
1930544        YSWF          MW  2025-10-07T23:55:00+10:00     11.60   

        unit_code_y  emission  
0                 t    0.0000  
1                 t    0.0000  
2                 t    0.0000  
3      

Here we :
- Take absolute value of Power since some Power is in -ve which cannot be true for a power plant

- drop redundant columns
- drop rows with 0 in both power and emission

In [58]:
power_emission_raw_stage_1['power_in_MW'] = power_emission_raw_stage_1['power_MW'].abs()
power_emission_raw_stage_1 = power_emission_raw_stage_1[~((power_emission_raw_stage_1['power_MW'] == 0) & (power_emission_raw_stage_1['emission'] == 0))]
power_emission_raw_stage_1 = power_emission_raw_stage_1.drop(columns=[col for col in ['unit_code_x', 'unit_code_y','power_MW'] if col in power_emission_raw_stage_1.columns])


print(power_emission_raw_stage_1.reset_index(drop=True))

        facility_id                  timestamp  emission  power_in_MW
0          0WAMBOWF  2025-10-01T00:00:00+10:00    0.0000       65.230
1               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
2               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
3               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
4               ADP  2025-10-01T00:00:00+10:00    0.0000        0.004
...             ...                        ...       ...          ...
1121424    YALLOURN  2025-10-07T23:55:00+10:00   39.0580        0.000
1121425      YAMBUK  2025-10-07T23:55:00+10:00    0.0000       14.600
1121426      YARWUN  2025-10-07T23:55:00+10:00    7.1024      138.990
1121427    YENDONWF  2025-10-07T23:55:00+10:00    0.0000       89.870
1121428        YSWF  2025-10-07T23:55:00+10:00    0.0000       11.600

[1121429 rows x 4 columns]


In [59]:

# Round to 3 decimal places (numeric precision)
power_emission_raw_stage_1["power_in_MW"] = power_emission_raw_stage_1["power_in_MW"].round(3)
power_emission_raw_stage_1["emission"] = power_emission_raw_stage_1["emission"].round(3)

# Format to always show exactly 3 decimal places
power_emission_raw_stage_1["power_in_MW"] = power_emission_raw_stage_1["power_in_MW"].map(lambda x: f"{x:.3f}")
power_emission_raw_stage_1["emission"] = power_emission_raw_stage_1["emission"].map(lambda x: f"{x:.3f}")

# Remove duplicates where timestamp, power, and emission are the same
power_emission_raw_stage_2 = power_emission_raw_stage_1.drop_duplicates(
    subset=["timestamp", "power_in_MW", "emission"]
).reset_index(drop=True)

power_emission_raw_stage_2

,facility_id,timestamp,emission,power_in_MW
0,0WAMBOWF,2025-10-01T00:00:00+10:00,0.000,65.230
1,ADP,2025-10-01T00:00:00+10:00,0.000,0.004
2,ARWF,2025-10-01T00:00:00+10:00,0.000,35.400
3,B2PS,2025-10-01T00:00:00+10:00,0.004,0.081
4,B2PS,2025-10-01T00:00:00+10:00,0.000,0.081
...,...,...,...,...
691514,YALLOURN,2025-10-07T23:55:00+10:00,39.058,360.562
691515,YAMBUK,2025-10-07T23:55:00+10:00,0.000,14.600
691516,YARWUN,2025-10-07T23:55:00+10:00,7.102,138.990
691517,YENDONWF,2025-10-07T23:55:00+10:00,0.000,89.870


## 2.2. Cleaning Facilities

In [60]:
facilities_df

,name,network_id,network_region,description,location,code,units,created_at,updated_at
0,Adelaide Desalination,NEM,SA1,"<p>The Adelaide Desalination plant (ADP), form...","{'lat': -35.096948, 'lng': 138.484061}",ADP,"[{'code': 'ADPPV1', 'fueltech_id': 'solar_util...",2023-10-18T04:34:30Z,2025-08-05T06:08:12Z
1,Albany,WEM,WEM,<p>Albany wind and Grasmere farms are two wind...,"{'lat': -35.065835, 'lng': 117.797506}",ALBANY,"[{'code': 'ALBANY_WF1', 'fueltech_id': 'wind',...",2023-10-18T04:34:30Z,2025-08-11T02:03:13Z
2,Aldoga,NEM,QLD1,<p>The Aldoga Solar Farm will be approximately...,"{'lat': -23.839544, 'lng': 151.0849}",ALDGASF,"[{'code': 'ALDGASF1', 'fueltech_id': 'solar_ut...",2025-01-31T04:19:33Z,2025-03-25T00:52:44Z
3,Amcor Glass,NEM,SA1,<p></p>,"{'lat': -34.882663, 'lng': 138.577975}",AMCORGR,"[{'code': 'AMCORGR', 'fueltech_id': 'distillat...",2023-10-18T04:34:32Z,2023-10-18T04:34:32Z
4,Angaston,NEM,SA1,<p>Angaston Power Station is a diesel-powered ...,"{'lat': -34.503948, 'lng': 139.024296}",ANGASTON,"[{'code': 'ANGAS1', 'fueltech_id': 'distillate...",2023-10-18T04:34:32Z,2025-09-07T01:53:13Z
...,...,...,...,...,...,...,...,...,...
575,Yarrawonga,NEM,VIC1,<p>Yarrawonga Power Station is adjacent to the...,"{'lat': -36.009466, 'lng': 145.999568}",YWNGAHYD,"[{'code': 'YWNGAHYD', 'fueltech_id': 'hydro', ...",2023-10-18T04:35:33Z,2024-10-29T22:19:53Z
576,Yarwun,NEM,QLD1,<p></p>,"{'lat': -23.8302, 'lng': 151.149692}",YARWUN,"[{'code': 'YARWUN_1', 'fueltech_id': 'gas_ccgt...",2023-10-18T04:35:33Z,2025-07-08T04:17:36Z
577,Yatpool,NEM,VIC1,<p>The 258-hectare Yatpool Solar Farm in north...,"{'lat': -34.38073, 'lng': 142.20534}",YATSF1,"[{'code': 'YATSF1', 'fueltech_id': 'solar_util...",2023-10-18T04:35:33Z,2025-07-08T05:20:08Z
578,Yawong,NEM,VIC1,<p>The Yawong Wind Farm was developed and cons...,"{'lat': -36.471022, 'lng': 143.361722}",YAWWF,"[{'code': 'YAWWF1', 'fueltech_id': 'wind', 'st...",2023-10-18T04:35:33Z,2024-10-30T23:59:54Z


In [62]:
# Step 1: Drop unwanted columns 
facilities_df_stage_1 = facilities_df.drop(columns=['description', 'created_at', 'updated_at'], errors='ignore')

# Step 2: Split 'location' dict into latitude and longitude
# If 'location' is a string, convert to dict first
facilities_df_stage_1['location'] = facilities_df_stage_1['location'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
facilities_df_stage_1['latitude'] = facilities_df_stage_1['location'].apply(lambda x: x.get('lat') if isinstance(x, dict) else None)
facilities_df_stage_1['longitude'] = facilities_df_stage_1['location'].apply(lambda x: x.get('lng') if isinstance(x, dict) else None)
facilities_df_stage_1 = facilities_df_stage_1.drop(columns=['location'])

#  Clean 'network_region' (remove trailing '1') 
facilities_df_stage_1['network_region'] = facilities_df_stage_1['network_region'].str.replace(r'1$', '', regex=True)

# Step 4: Extract first 'fueltech_id' from 'units' list 
# Convert stringified lists to Python lists (if needed)
facilities_df_stage_1['units'] = facilities_df_stage_1['units'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
facilities_df_stage_1['fuel_source'] = facilities_df_stage_1['units'].apply(
    lambda x: x[0]['fueltech_id'] if isinstance(x, list) and len(x) > 0 else None
)

# Optional: Drop original 'units' column if no longer needed
#facilities_df_stage_1 = facilities_df_stage_1.drop(columns=['units'])

# Final cleaned DataFrame
print(facilities_df_stage_1)

                      name network_id network_region      code  \
0    Adelaide Desalination        NEM             SA       ADP   
1                   Albany        WEM            WEM    ALBANY   
2                   Aldoga        NEM            QLD   ALDGASF   
3              Amcor Glass        NEM             SA   AMCORGR   
4                 Angaston        NEM             SA  ANGASTON   
..                     ...        ...            ...       ...   
575             Yarrawonga        NEM            VIC  YWNGAHYD   
576                 Yarwun        NEM            QLD    YARWUN   
577                Yatpool        NEM            VIC    YATSF1   
578                 Yawong        NEM            VIC     YAWWF   
579                 Yendon        NEM            VIC  YENDONWF   

                                                 units   latitude   longitude  \
0    [{'code': 'ADPPV1', 'fueltech_id': 'solar_util... -35.096948  138.484061   
1    [{'code': 'ALBANY_WF1', 'fueltech_id': '

## 2.3. Combine facilities and power-emission

Left-Join facility attributes (name, region, location etc.) onto the power-emissions data

In [63]:
power_emission_raw_stage_3 = pd.merge(
    power_emission_raw_stage_2,
    facilities_df_stage_1,
    how='left',
    left_on='facility_id',
    right_on='code'
)

In [64]:
power_emission_raw_stage_3

,facility_id,timestamp,emission,power_in_MW,name,network_id,network_region,code,units,latitude,longitude,fuel_source
0,0WAMBOWF,2025-10-01T00:00:00+10:00,0.000,65.230,Wambo,NEM,QLD,0WAMBOWF,"[{'code': '0WAMBOWF2', 'fueltech_id': 'wind', ...",-26.603045,151.246876,wind
1,ADP,2025-10-01T00:00:00+10:00,0.000,0.004,Adelaide Desalination,NEM,SA,ADP,"[{'code': 'ADPPV1', 'fueltech_id': 'solar_util...",-35.096948,138.484061,solar_utility
2,ARWF,2025-10-01T00:00:00+10:00,0.000,35.400,Ararat,NEM,VIC,ARWF,"[{'code': 'ARWF1', 'fueltech_id': 'wind', 'sta...",-37.263393,143.082116,wind
3,B2PS,2025-10-01T00:00:00+10:00,0.004,0.081,Braemar 2,NEM,QLD,B2PS,"[{'code': 'BRAEMAR7', 'fueltech_id': 'gas_ocgt...",-27.109873,150.907653,gas_ocgt
4,B2PS,2025-10-01T00:00:00+10:00,0.000,0.081,Braemar 2,NEM,QLD,B2PS,"[{'code': 'BRAEMAR7', 'fueltech_id': 'gas_ocgt...",-27.109873,150.907653,gas_ocgt
...,...,...,...,...,...,...,...,...,...,...,...,...
691514,YALLOURN,2025-10-07T23:55:00+10:00,39.058,360.562,Yallourn W,NEM,VIC,YALLOURN,"[{'code': 'YWPS1', 'fueltech_id': 'coal_brown'...",-38.177596,146.347508,coal_brown
691515,YAMBUK,2025-10-07T23:55:00+10:00,0.000,14.600,Yambuk,NEM,VIC,YAMBUK,"[{'code': 'YAMBUKWF', 'fueltech_id': 'wind', '...",-38.305278,142.018439,wind
691516,YARWUN,2025-10-07T23:55:00+10:00,7.102,138.990,Yarwun,NEM,QLD,YARWUN,"[{'code': 'YARWUN_1', 'fueltech_id': 'gas_ccgt...",-23.830200,151.149692,gas_ccgt
691517,YENDONWF,2025-10-07T23:55:00+10:00,0.000,89.870,Yendon,NEM,VIC,YENDONWF,"[{'code': 'YENDWF1', 'fueltech_id': 'wind', 's...",-37.630952,144.022463,wind


In [65]:
power_emission_raw_stage_4 = power_emission_raw_stage_3.drop(columns=["network_id", "code"])
print(power_emission_raw_stage_4)

       facility_id                  timestamp emission power_in_MW  \
0         0WAMBOWF  2025-10-01T00:00:00+10:00    0.000      65.230   
1              ADP  2025-10-01T00:00:00+10:00    0.000       0.004   
2             ARWF  2025-10-01T00:00:00+10:00    0.000      35.400   
3             B2PS  2025-10-01T00:00:00+10:00    0.004       0.081   
4             B2PS  2025-10-01T00:00:00+10:00    0.000       0.081   
...            ...                        ...      ...         ...   
691514    YALLOURN  2025-10-07T23:55:00+10:00   39.058     360.562   
691515      YAMBUK  2025-10-07T23:55:00+10:00    0.000      14.600   
691516      YARWUN  2025-10-07T23:55:00+10:00    7.102     138.990   
691517    YENDONWF  2025-10-07T23:55:00+10:00    0.000      89.870   
691518        YSWF  2025-10-07T23:55:00+10:00    0.000      11.600   

                         name network_region  \
0                       Wambo            QLD   
1       Adelaide Desalination             SA   
2              

For Final Cleanup, we drop duplicates

In [66]:

power_emission_final = power_emission_raw_stage_4.drop_duplicates(
    subset=['facility_id', 'timestamp'],
    keep='first'  # or 'last' if you want to keep the last occurrence
)

# Optional: reset index
power_emission_final.reset_index(drop=True, inplace=True)

## 2.4. Price and Demand Cleanup

Clean Up Per Market Price and Demand

In [70]:
market_raw_stage_1 =market_raw

In [71]:
# Remove the trailing '1' from region codes (e.g., 'NSW1' to 'NSW')
market_raw_stage_1['network_region'] = market_raw_stage_1['network_region'].str.replace(r'1$', '', regex=True)

In [72]:
# Remove duplicate timestamp rows to avoid double counting price/demand
market_raw_stage_1 = market_raw_stage_1.drop_duplicates(
    subset=["timestamp", "value_price", "value_demand"]
).reset_index(drop=True)


# Drop network column (not needed after region-level aggregation)
market_raw_stage_2 = market_raw_stage_1.drop(columns=["network"])

# Rename metric columns and drop unit columns since units are implied
market_raw_stage_2 = market_raw_stage_2.rename(
    columns={
        "value_price": "value_price_in_$/MWh",
        "value_demand": "value_demand_in_MW"
    }
).drop(columns=["unit_price", "unit_demand"])
# Force all price and demand values to positive (market data sometimes returns signed numbers)
market_raw_stage_2["value_price_in_$/MWh"] = market_raw_stage_2["value_price_in_$/MWh"].abs()
market_raw_stage_2["value_demand_in_MW"] = market_raw_stage_2["value_demand_in_MW"].abs()

In [73]:
# Sort market data chronologically and by region for consistent merging
market_raw_stage_3 = market_raw_stage_2.sort_values(
    by=["timestamp", "network_region"]
).reset_index(drop=True)

## 2.5. Merging Power and Market Datasets

In [75]:
# Merge facility time-series data with market price & demand based on timestamp and region
merged_df = pd.merge(
    power_emission_final,market_raw_stage_3
    [["timestamp", "network_region", "value_price_in_$/MWh", "value_demand_in_MW"]],
    on=["timestamp", "network_region"],
    how="left"
)

# Quick sanity check
print("Merged shape:", merged_df.shape)
merged_df.head(5)

Merged shape: (406845, 12)


,facility_id,timestamp,emission,power_in_MW,name,network_region,units,latitude,longitude,fuel_source,value_price_in_$/MWh,value_demand_in_MW
0,0WAMBOWF,2025-10-01T00:00:00+10:00,0.000,65.230,Wambo,QLD,"[{'code': '0WAMBOWF2', 'fueltech_id': 'wind', ...",-26.603045,151.246876,wind,54.82,5989.24
1,ADP,2025-10-01T00:00:00+10:00,0.000,0.004,Adelaide Desalination,SA,"[{'code': 'ADPPV1', 'fueltech_id': 'solar_util...",-35.096948,138.484061,solar_utility,8.11,1564.92
2,ARWF,2025-10-01T00:00:00+10:00,0.000,35.400,Ararat,VIC,"[{'code': 'ARWF1', 'fueltech_id': 'wind', 'sta...",-37.263393,143.082116,wind,8.95,4893.49
3,B2PS,2025-10-01T00:00:00+10:00,0.004,0.081,Braemar 2,QLD,"[{'code': 'BRAEMAR7', 'fueltech_id': 'gas_ocgt...",-27.109873,150.907653,gas_ocgt,54.82,5989.24
4,BANGOWF,2025-10-01T00:00:00+10:00,0.000,97.699,Bango,NSW,"[{'code': 'BANGOWF1', 'fueltech_id': 'wind', '...",-34.767208,148.921499,wind,56.98,7105.57


In [76]:
# Save to a new CSV for use in Stage 3
merged_df.to_csv("power_emission.csv", index=False)

# 3. Data Publishing via MQTT

## 3.1. MQTT configs and setup

In [2]:
MQTT_BROKER = "test.mosquitto.org"
MQTT_PORT   = 1883
TOPIC_POWER = "470461201/A2"  # topic for pub./sub
PUBLISH_DELAY = 0.1         # seconds between messages  
RELOAD_DELAY  = 0          # seconds to wait between rounds when REPEAT=True
END_WAIT_SECS = 60           # wait before replay restart
REPEAT = True                # keep publishing in a loop

## 3.2. Transforming Facilities Unit Data

Parsing and Extracting Unit Codes from Facility Metadata

In [3]:
def parse_units_codes(units_cell):
    """Return ONLY the 'code' values from the 'units' JSON."""
    if units_cell is None or (isinstance(units_cell, float) and pd.isna(units_cell)):
        return []
    obj = units_cell
    # If stored as a string (JSON or Python literal), try to parse it
    if isinstance(units_cell, str):
        s = units_cell.strip()
        if not s:
            return []
        try:
            obj = json.loads(s) #JSON
        except Exception:
            try:
                obj = ast.literal_eval(s) #Python
            except Exception:
                return []
    # Ensure we are working with a list of dicts
    if isinstance(obj, dict):
        obj = [obj]
    if not isinstance(obj, (list, tuple)):
        return []
    return [str(d.get("code")) for d in obj if isinstance(d, dict) and d.get("code")]

CSV_PATH = "power_emission.csv"
power_df = pd.read_csv(CSV_PATH)

# Extract clean list of codes and a display-friendly string
power_df["unit_codes"] = power_df["units"].apply(parse_units_codes)
power_df["unit_codes_str"] = power_df["unit_codes"].apply(lambda xs: ", ".join(xs) if xs else "—")

display(power_df[["facility_id","name","unit_codes","unit_codes_str"]].head(5))
# Ensure timestamps are datetime objects
power_df["timestamp"] = pd.to_datetime(power_df["timestamp"])






,facility_id,name,unit_codes,unit_codes_str
0,0WAMBOWF,Wambo,"[0WAMBOWF2, WAMBOWF1]","0WAMBOWF2, WAMBOWF1"
1,ADP,Adelaide Desalination,"[ADPPV1, ADPPV2, ADPPV3, ADPBA1G, ADPBA1L, ADP...","ADPPV1, ADPPV2, ADPPV3, ADPBA1G, ADPBA1L, ADPBA1"
2,ARWF,Ararat,[ARWF1],ARWF1
3,B2PS,Braemar 2,"[BRAEMAR7, BRAEMAR5, BRAEMAR6]","BRAEMAR7, BRAEMAR5, BRAEMAR6"
4,BANGOWF,Bango,"[BANGOWF1, BANGOWF2]","BANGOWF1, BANGOWF2"


Initializing MQTT Client and Publish Callback

In [4]:
client = mqtt.Client()
client.connect(MQTT_BROKER, MQTT_PORT, 60)
client.loop_start()

def on_publish(client, userdata, mid):
    print(f"MQTT message with mid={mid} successfully published.")

client.on_publish = on_publish

/var/folders/tv/ly87y_cd5vzcqrw28_rhj9t40000gn/T/ipykernel_37832/3426122455.py:1: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


## 3.3. Publisher Setup (5. Continuous Execution)

Publishing Time-Series Power and Emissions Data as MQTT Stream. Duplicaton Avoided and Continuous Execution Executed

In [5]:
def publish_stream(power_df, delay_between_messages=PUBLISH_DELAY, delay_between_timestamps=RELOAD_DELAY):
    all_timestamps = sorted(power_df["timestamp"].unique())

    #remember last (power, emission) per facility for the current run
    last_seen = {}  # facility_id -> (power_val, emission_val)
    EPS = 1e-6      # small tolerance for float equality

    for ts in all_timestamps:
        print(f"\nPublishing data for timestamp: {ts}")

        # Pulling and Sending Data
        subset_power = power_df[power_df["timestamp"] == ts]
        for _, row in subset_power.iterrows():
            # Parse values
            power_val     = float(row.get("power_in_MW", 0) or 0)
            emission_val  = float(row.get("emission",   0) or 0)
            facility_id   = row.get("facility_id")

            # Skip rows with both zero
            if power_val == 0 and emission_val == 0:
                continue

            #skip if duplicate vs previous timestamp for the same facility
            prev = last_seen.get(facility_id)
            if prev is not None:
                p_prev, e_prev = prev
                if math.isclose(power_val, p_prev, rel_tol=0.0, abs_tol=EPS) and \
                   math.isclose(emission_val, e_prev, rel_tol=0.0, abs_tol=EPS):

                    continue

            # Message Data
            msg_power = {
                "facility_id": row["facility_id"],
                "name": row["name"],
                "timestamp": str(row["timestamp"]),
                "power_MW": power_val,
                "emissions_tCO2": emission_val,
                "network_region": row["network_region"],
                "fuel_source": row["fuel_source"],
                "latitude": float(row["latitude"]),
                "longitude": float(row["longitude"]),
                "value_price_in_$/MWh": float(row.get("value_price_in_$/MWh")),
                "value_demand_in_MW": float(row.get("value_demand_in_MW")),
                "unit_codes": row.get("unit_codes"),
                "unit_codes_str": row.get("unit_codes_str"),
            }

            client.publish(TOPIC_POWER, json.dumps(msg_power))
            print(f"  Power published: {msg_power['facility_id']} ({msg_power['power_MW']} MW)")

            #update last seen AFTER a publish
            last_seen[facility_id] = (power_val, emission_val)

            # Delay between messages is 0.1s
            time.sleep(delay_between_messages)

        # Delay between timestamp batches
        print(f"Finished timestamp {ts}. Waiting {delay_between_timestamps}s before next batch...")
        time.sleep(delay_between_timestamps)

    # Continuous Execution (replay)
    if REPEAT:
        print(f"End of data reached. Waiting {END_WAIT_SECS}s before restarting...")
        time.sleep(END_WAIT_SECS)
        #call the function again to simulate unbounded data
        publish_stream(power_df)
    return

## 3.4. Running Publisher

Publishing the Data

In [6]:
try:
    publish_stream(power_df)
except KeyboardInterrupt:
    print("\n Streaming stopped by user.")
finally:
    client.loop_stop()
    client.disconnect()
    print(" MQTT client disconnected")



Publishing data for timestamp: 2025-10-01 00:00:00+10:00
  Power published: 0WAMBOWF (65.23 MW)
MQTT message with mid=1 successfully published.
  Power published: ADP (0.004 MW)
MQTT message with mid=2 successfully published.
  Power published: ARWF (35.4 MW)
MQTT message with mid=3 successfully published.
  Power published: B2PS (0.081 MW)
MQTT message with mid=4 successfully published.
  Power published: BANGOWF (97.699 MW)
MQTT message with mid=5 successfully published.
  Power published: BARCSF (0.1 MW)MQTT message with mid=6 successfully published.

  Power published: BASTYAN (57.1 MW)MQTT message with mid=7 successfully published.

  Power published: BAYSW (558.395 MW)MQTT message with mid=8 successfully published.

  Power published: BBATTERY (0.015 MW)
MQTT message with mid=9 successfully published.
  Power published: BHB (2.194 MW)
MQTT message with mid=10 successfully published.
  Power published: BHWF (7.764 MW)MQTT message with mid=11 successfully published.

  Power publi

  Power published: MACARTH (126.2 MW)
MQTT message with mid=95 successfully published.
  Power published: MACKNTSH (51.81 MW)
MQTT message with mid=96 successfully published.
  Power published: MANNUMB (0.18 MW)
MQTT message with mid=97 successfully published.
  Power published: MBAHNTH (20.2 MW)
MQTT message with mid=98 successfully published.
  Power published: MCINTYR (139.076 MW)
MQTT message with mid=99 successfully published.
  Power published: MCKAY (0.02 MW)MQTT message with mid=100 successfully published.

  Power published: MEADOWBK (43.442 MW)MQTT message with mid=101 successfully published.

  Power published: MEWF (49.5 MW)
MQTT message with mid=102 successfully published.
  Power published: MILLMERN (0.0 MW)MQTT message with mid=103 successfully published.

  Power published: MINTARO (0.085 MW)
MQTT message with mid=104 successfully published.
  Power published: MLSP (0.013 MW)
MQTT message with mid=105 successfully published.
  Power published: MLWF (11.55 MW)
MQTT messa

  Power published: BOCOROCK (53.98 MW)MQTT message with mid=189 successfully published.

  Power published: BODWF (44.137 MW)
MQTT message with mid=190 successfully published.
  Power published: BOLIVAR (0.003 MW)
MQTT message with mid=191 successfully published.
  Power published: BRYB1WF1 (129.149 MW)
MQTT message with mid=192 successfully published.
  Power published: BULGANA (47.56 MW)MQTT message with mid=193 successfully published.

  Power published: BUTLERSG (6.5 MW)
MQTT message with mid=194 successfully published.
  Power published: BWTR1 (23.836 MW)MQTT message with mid=195 successfully published.

  Power published: CALLIDEC1 (0.0 MW)MQTT message with mid=196 successfully published.

  Power published: CANUNDA1 (34.74 MW)
MQTT message with mid=197 successfully published.
  Power published: CAPBES (0.94 MW)
MQTT message with mid=198 successfully published.
  Power published: CAPTL_WF (115.014 MW)MQTT message with mid=199 successfully published.

  Power published: CATHROCK (

  Power published: PIONEER (42.7 MW)
MQTT message with mid=283 successfully published.
  Power published: PORTWF (55.9 MW)
MQTT message with mid=284 successfully published.
  Power published: PTINA (3.32 MW)
MQTT message with mid=285 successfully published.
  Power published: RANGEB (0.57 MW)MQTT message with mid=286 successfully published.

  Power published: REECE (118.764 MW)
MQTT message with mid=287 successfully published.
  Power published: REPULSE (33.013 MW)MQTT message with mid=288 successfully published.

  Power published: RESS (0.695 MW)MQTT message with mid=289 successfully published.

  Power published: RIVNB (0.869 MW)
MQTT message with mid=290 successfully published.
  Power published: RYANCWF (74.32 MW)
MQTT message with mid=291 successfully published.
  Power published: RYEPARK (184.803 MW)
MQTT message with mid=292 successfully published.
  Power published: SALTCRK (14.06 MW)
MQTT message with mid=293 successfully published.
  Power published: SAPHWF1 (232.882 MW)MQT

  Power published: ELAINEWF (45.19 MW)MQTT message with mid=375 successfully published.

  Power published: ERARING (455.293 MW)
MQTT message with mid=376 successfully published.
  Power published: ERB (1.111 MW)
MQTT message with mid=377 successfully published.
  Power published: FISHER (44.77 MW)MQTT message with mid=378 successfully published.

  Power published: FLYCRKWF (9.95 MW)
MQTT message with mid=379 successfully published.
  Power published: GANNBESS (0.206 MW)
MQTT message with mid=380 successfully published.
  Power published: GERMCRK (17.189 MW)MQTT message with mid=381 successfully published.

  Power published: GORDON (24.24 MW)
MQTT message with mid=382 successfully published.
  Power published: GPWFEST (175.14 MW)MQTT message with mid=383 successfully published.

  Power published: GRANWF (29.67 MW)
MQTT message with mid=384 successfully published.
  Power published: GREENB (0.243 MW)MQTT message with mid=385 successfully published.

  Power published: GSTONE (150.199

  Power published: WAUBRAWF (59.846 MW)
MQTT message with mid=469 successfully published.
  Power published: WDBESS (0.548 MW)
MQTT message with mid=470 successfully published.
  Power published: WGWF (104.5 MW)
MQTT message with mid=471 successfully published.
  Power published: WKIEWA (17.0 MW)
MQTT message with mid=472 successfully published.
  Power published: WOODLWN (13.452 MW)
MQTT message with mid=473 successfully published.
  Power published: WOOLNTH1 (25.48 MW)MQTT message with mid=474 successfully published.

  Power published: WPWF (45.74 MW)
MQTT message with mid=475 successfully published.
  Power published: WRWF1 (42.274 MW)
MQTT message with mid=476 successfully published.
  Power published: WTAHB (1.777 MW)
MQTT message with mid=477 successfully published.
  Power published: YALLOURN (246.25 MW)MQTT message with mid=478 successfully published.

  Power published: YAMBUK (5.3 MW)
MQTT message with mid=479 successfully published.
  Power published: YARWUN (85.01 MW)MQTT 

  Power published: LKBONNY3 (25.734 MW)
MQTT message with mid=561 successfully published.
  Power published: LKBONNY_2 (79.501 MW)
MQTT message with mid=562 successfully published.
  Power published: LOYYANGA (372.0 MW)
MQTT message with mid=563 successfully published.
  Power published: LOYYB (454.125 MW)
MQTT message with mid=564 successfully published.
  Power published: MACARTH (100.6 MW)
MQTT message with mid=565 successfully published.
  Power published: MACKNTSH (69.15 MW)MQTT message with mid=566 successfully published.

  Power published: MANNUMB (4.77 MW)
MQTT message with mid=567 successfully published.
  Power published: MCINTYR (139.575 MW)MQTT message with mid=568 successfully published.

  Power published: MEADOWBK (43.442 MW)MQTT message with mid=569 successfully published.

  Power published: MEWF (48.29 MW)
MQTT message with mid=570 successfully published.
  Power published: MILLMERN (0.0 MW)
MQTT message with mid=571 successfully published.
  Power published: MLWF (1

  Power published: CETHANA (67.469 MW)
MQTT message with mid=655 successfully published.
  Power published: CHBESS (0.135 MW)MQTT message with mid=656 successfully published.

  Power published: CHYTWF (14.55 MW)
MQTT message with mid=657 successfully published.
  Power published: CLEMGPWF (48.48 MW)
MQTT message with mid=658 successfully published.
  Power published: CLRKCWF (265.011 MW)MQTT message with mid=659 successfully published.

  Power published: CLUNY (18.24 MW)
MQTT message with mid=660 successfully published.
  Power published: COLWF01 (135.491 MW)
MQTT message with mid=661 successfully published.
  Power published: CONDONG1 (19.69 MW)
MQTT message with mid=662 successfully published.
  Power published: COOPGWF (327.54 MW)MQTT message with mid=663 successfully published.

  Power published: CROOKWF2 (75.964 MW)
MQTT message with mid=664 successfully published.
  Power published: CROOKWF3 (33.43 MW)
MQTT message with mid=665 successfully published.
  Power published: CROWLW

  Power published: SMTHBES1 (0.066 MW)MQTT message with mid=749 successfully published.

  Power published: SNOWNTH (122.2 MW)
MQTT message with mid=750 successfully published.
  Power published: SNOWSTH (111.9 MW)
MQTT message with mid=751 successfully published.
  Power published: SNOWTOWN (91.663 MW)
MQTT message with mid=752 successfully published.
  Power published: SNOWY3 (61.4 MW)
MQTT message with mid=753 successfully published.
  Power published: STANWELL (265.38 MW)
MQTT message with mid=754 successfully published.
  Power published: STARFHILL (6.298 MW)MQTT message with mid=755 successfully published.

  Power published: STOCKYD (331.52 MW)
MQTT message with mid=756 successfully published.
  Power published: STWF (147.433 MW)
MQTT message with mid=757 successfully published.
  Power published: TARALGA (46.024 MW)MQTT message with mid=758 successfully published.

  Power published: TARONGN (277.938 MW)
MQTT message with mid=759 successfully published.
  Power published: TARRA

  Power published: GREENB (0.054 MW)MQTT message with mid=843 successfully published.

  Power published: GSTONE (110.086 MW)MQTT message with mid=844 successfully published.

  Power published: GSWF (201.11 MW)
MQTT message with mid=845 successfully published.
  Power published: GULLRGWF (119.113 MW)
MQTT message with mid=846 successfully published.
  Power published: GUNNING (35.138 MW)MQTT message with mid=847 successfully published.

  Power published: HALLWF (74.07 MW)
MQTT message with mid=848 successfully published.
  Power published: HALLWF2 (56.19 MW)MQTT message with mid=849 successfully published.

  Power published: HD1WF (46.56 MW)MQTT message with mid=850 successfully published.

  Power published: HORNSDAL (76.7 MW)MQTT message with mid=851 successfully published.

  Power published: HORNSDAL2 (93.9 MW)
MQTT message with mid=852 successfully published.
  Power published: HORNSDAL3 (91.9 MW)
MQTT message with mid=853 successfully published.
  Power published: HORNSDPR (6.

# Validation

Chicking unique facilities and how many have a singular type of fuel type in units 

In [3]:
df = pd.read_csv("power_emission.csv")

#Fuel Groups
fuel_groups = {
    'battery': 'battery',
    'battery_charging': 'battery',
    'battery_discharging': 'battery',
    'bidirectional': 'battery',
    
    'solar_utility': 'solar',
    'solar_rooftop': 'solar',
    'solar': 'solar',

    'wind': 'wind',
    'wind_offshore': 'wind',

    'coal_black': 'coal',
    'coal_brown': 'coal',

    'gas_ocgt': 'gas',
    'gas_ccgt': 'gas',
    'gas': 'gas',
}

# Convert the 'units' column from a string representation of a list-of-dicts
# into an actual Python list-of-dicts so we can inspect fields inside it.
df['units_parsed'] = df['units'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])

# Function to check if all units inside a facility share the *same fuel group*
def unified_fueltech(units_list):
    ft = [fuel_groups.get(u.get('fueltech_id'), u.get('fueltech_id')) 
          for u in units_list if isinstance(u, dict)]
    return len(set(ft)) == 1

# Apply the uniformity check to each facility row.
df['all_same_grouped'] = df['units_parsed'].apply(unified_fueltech)

# Collapse dataset to one record per facility (avoid duplicated facility rows).
result_grouped = df[['name', 'all_same_grouped']].drop_duplicates()

# Count how many facilities are uniform vs how many total.
num_same_grouped = result_grouped['all_same_grouped'].sum()
num_total = result_grouped.shape[0]

print(f"Facilities uniform after grouping: {num_same_grouped} out of {num_total}")

Facilities uniform after grouping: 287 out of 304
